In [ ]:
from IPython.display import clear_output

!pip install roboflow adversarial-robustness-toolbox huggingface_hub

clear_output(wait=True)
print("Installed successfully")

In [ ]:
# Clone detectron2
import sys, os, distutils.core

!python -m pip install pyyaml==5.1
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

clear_output(wait=True)
print("Installed successfully")

In [ ]:
import time
import numpy as np 
import torch
import torch.nn as nn
import logging
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import time
import cv2
import random

from PIL import Image
from torchvision import transforms
import torch.optim as optim
import torchvision.models as models
from torchvision.models import resnet50
import matplotlib.pyplot as plt 

# Adversarial Robustness Toolbox
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent
from skimage.metrics import structural_similarity as ssim 

# Detectron2
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

# Roboflow
from roboflow import Roboflow

import warnings
warnings.filterwarnings('ignore') 

logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

In [ ]:
rf = Roboflow(api_key="Zykmbeiu1pSoB9Tzeom8")
project = rf.workspace("thesis-vrpuo").project("lspd-hod-ozsk1")
version = project.version(2)
dataset = version.download("coco")

clear_output(wait=True)
print("Installed successfully")

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scale=2
THRESHOLD = 0.5
EPS = 0.02

PROCESSOR = ViltProcessor.from_pretrained("dandelin/vilt-b32-finetuned-vqa")
MODEL = ViltForQuestionAnswering.from_pretrained("dandelin/vilt-b32-finetuned-vqa")

# Valid file extension
EXT = (".jpg", ".jpeg", ".png", ".bmp", ".gif")

# CHECKPOINTS
CHECKPOINT_FRCNN = "/kaggle/input/current-active-model/pytorch/default/1/model_final.pth"
CHECKPOINT_REALESRGAN = "/kaggle/input/current-active-model/pytorch/default/1/RealESRGAN_x2.pth"

# Number of Classes
NUM_CLASSES = 11

# Classes
COCO_CLASSES = {
    1: "breast",
    2: "anus",
    3: "female_genital",
    4: "male_genital",
    5: "harmful_object",
    6: "safe",
    7: "self_harm",
    8: "sexual_content",
    9: "toxic_substance",
    10: "violence"
}

QUESTIONS = [
    # --- Context ---
    "Is this a painting?",
    "Is this for educational purposes?",

    # --- Nudity & Sexual Content ---
    "Are people engaging in a sexual act?",

    # --- Violence & Weapons ---
    "Is there a weapon?",
    "Is there an injured person?",
    "Is there a dead person?",
    "Are people fighting?",
    "Is one person threatening another person?",

    # --- Other Harms ---
    "Are there drugs?",
    "Is there alcohol?",
    "Is there a cigarette?",
    "Is there a fire?"
]

clear_output(wait=True)
print("Executed successfully")

# Attacked Data
* This portion will add PGD and FGSM attack on test dataset. 

In [ ]:
def softmax_activation(inputs): 
    inputs = inputs.tolist()
    exp_values = np.exp(inputs - np.max(inputs)) 
    # Normalize 
    probabilities = exp_values / np.sum(exp_values)
    return probabilities 

## Comparison on image
* This used PSNR (Peak Signal Noise Ratio) and SSIM.
* If PSNR is less than 25, means it degraded the image.
* If PSNR is between 25-30 and above, meaning the image has been purified in moderately.
* If SSIM is lower than 50, means that there is a loss in visual representation.

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor()
]) 

In [ ]:
classes = {
    0: "background",
    1: "alcohol",
    2: "anus",
    3: "blood",
    4: "breast",
    5: "cigarette",
    6: "female_genital",
    7: "gun",
    8: "insulting_gesture",
    9: "knife",
    10: "male_genital"
}

In [ ]:
model_resnet50 = models.resnet50(pretrained=True)  
criterion = nn.CrossEntropyLoss()

# Create the ART classifier
classifier = PyTorchClassifier(
    model=model_resnet50,
    loss=criterion,
    input_shape=(3, 224, 224),
    nb_classes=1000,
    device_type='gpu'
)

In [ ]:
input_folder_path = "/kaggle/working/lspd+hod-2/test"
# input_folder_path = "/kaggle/working/lspd+hod-2/valid"
# input_folder_path = "/kaggle/working/test"
fgsm_folder_path = "/kaggle/working/fgsm"
valid_ext = (".jpg", ".jpeg", ".png", ".bmp", ".gif")

os.makedirs(fgsm_folder_path, exist_ok=True)

## Adding adversarial patches on data

In [ ]:
# FGSM
for filename in os.listdir(input_folder_path):
    if filename.lower().endswith(valid_ext):
        img_path = os.path.join(input_folder_path, filename)
        print(f"Processing: {filename}")

        # Open image
        input_image = Image.open(img_path).convert("RGB")
        original_size = input_image.size

        # Preprocess -> tensor -> batch
        input_tensor = preprocess(input_image)
        input_batch = input_tensor.unsqueeze(0).numpy().astype(np.float32)

        # Convert to HWC format for visualization
        input_vis = input_batch[0].transpose((1, 2, 0))

        preds = classifier.predict(input_batch)
        print(np.argmax(preds, axis=1))
        
        accuracy = np.max(softmax_activation(preds), axis=1)
        accuracy = round(accuracy[0], 2)
        print("Accuracy on benign examples: {}%".format(accuracy * 100))
        
        fgsm_attack = FastGradientMethod(estimator = classifier, eps=EPS) 

        start = time.time()
        x_test_adv = fgsm_attack.generate(x=input_batch)
        print("Time for attack (in seconds): {}".format(time.time()-start))
        
        predictions = classifier.predict(x_test_adv)
        print(np.argmax(predictions, axis=1))
        
        accuracy = round(np.max(softmax_activation(predictions), axis=1)[0]*100,2)
        print("Accuracy on adversarial test examples: {}%".format(accuracy))

        adv_img = x_test_adv[0].transpose((1,2,0))

        # Display
        # plt.title("FGSM")
        # plt.imshow(adv_img)
        # plt.show()

        fgsm_output_path = os.path.join(fgsm_folder_path, filename)
        
        adv_img_clip = np.clip(adv_img, 0, 1)
        adv_img_pil = Image.fromarray((adv_img_clip*255).astype(np.uint8))
        adv_img_resized = adv_img_pil.resize(original_size, Image.LANCZOS)
        adv_img_resized.save(fgsm_output_path)   

clear_output(wait=True)
print("PGD process success")

## Inference on attacked data
* This section will show the comparison between the prediction and ground truths under adversarial attacks.
* It will demonstrate whether Faster R-CNN is capable of detecting objects even when subjected to adversarial attacks like PGD and FGSM.

In [ ]:
register_coco_instances(
    "test",
    {},
    "/kaggle/working/lspd+hod-2/test/_annotations.coco.json",
    "/kaggle/working/lspd+hod-2/test"
)

register_coco_instances(
    "test_fgsm",
    {},
    "/kaggle/working/lspd+hod-2/test/_annotations.coco.json",
    "/kaggle/working/fgsm"
)


In [ ]:
NUM_CLASSES = 11
cfg = get_cfg()

cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = "/kaggle/input/current-active-model/pytorch/default/1/model_final.pth"

cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = THRESHOLD

# Comment this if GPU is available
os.environ["CUDA_VISIBLE_DEVICES"] = ""

cfg.MODEL.DEVICE = "cpu"

predictor = DefaultPredictor(cfg)
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

In [ ]:
def ground_pred(dataset):
    fig, axs = plt.subplots(2, 2, figsize=(12, 8))
    
    dataset_metadata = MetadataCatalog.get(dataset)
    dataset_dicts = DatasetCatalog.get(dataset)

    samples = random.sample(dataset_dicts, 2)
    
    for i, data in enumerate(samples):
        img = cv2.imread(data["file_name"])

        # Ground truth
        visualizer_truth = Visualizer(img[:, :, ::-1], metadata = dataset_metadata, scale = 1.2)
        visualize_truth = visualizer_truth.draw_dataset_dict(data)

        # Prediction
        outputs = predictor(img)
        visualizer_pred = Visualizer(img[:, :, ::-1], metadata = dataset_metadata, scale = 1.2)
        visualize_pred = visualizer_pred.draw_instance_predictions(outputs["instances"].to("cpu"))

        axs[i, 0].imshow(cv2.cvtColor(visualize_truth.get_image()[:, :, ::-1], cv2.COLOR_BGR2RGB))
        axs[i, 0].set_title("Ground Truth")
        axs[i, 0].axis('off')
        
        axs[i, 1].imshow(cv2.cvtColor(visualize_pred.get_image()[:, :, ::-1], cv2.COLOR_BGR2RGB))
        axs[i, 1].set_title("Prediction")
        axs[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

## Prediction, AP, and mAP of attacked data

In [ ]:
# Test Dataset
evaluator = COCOEvaluator("test", cfg, False, output_dir="./output/")
test_loader = build_detection_test_loader(cfg, "test")  # test dataset

results = inference_on_dataset(predictor.model, test_loader, evaluator)

map_cln = results["bbox"]["AP"]

print(f"Validation Set Metrics\n{results}")
print(f"mAP: {map_cln:.4f}")

ground_pred("test")

In [ ]:
# Test FGSM Dataset
evaluator = COCOEvaluator("test_fgsm", cfg, False, output_dir="./output/")
test_loader = build_detection_test_loader(cfg, "test_fgsm")  # test dataset

results = inference_on_dataset(predictor.model, test_loader, evaluator)

map_fgsm = results["bbox"]["AP"]

print(f"Validation Set Metrics\n{results}")
print(f"mAP: {map_fgsm:.4f}")

ground_pred("test_fgsm")

# Purifying Data
* Our tool conducted a purification for adversarial examples like FGSM and PGD using anisotropic diffusion and Real-ESRGAN. Now we will test whether it effectively removes the adversarial attack and if the classifier can effectively detect the right object. 

In [ ]:
# Clone Real-ESRGAN
!git clone 'https://github.com/ai-forever/Real-ESRGAN'

In [ ]:
# Move real-esrgan from parent folder
# Delete cloned folder
# Rename child folder
import shutil

parent_folder = "/kaggle/working/Real-ESRGAN/RealESRGAN"
child_folder = "/kaggle/working/RealESRGAN"
input_path = "/kaggle/working/Real-ESRGAN"
init_file = "/kaggle/working/real_esrgan/__init__.py"
# input_path = "/kaggle/working/test"
new_name = "/kaggle/working/real_esrgan"

shutil.move(parent_folder, child_folder)

if os.path.exists(input_path):
    shutil.rmtree(input_path)
    print("Folder deleted successfully")

os.rename(child_folder, new_name)
sys.path.insert(0, os.path.abspath('./real_esrgan'))
os.remove(init_file)

In [ ]:
import torch
import torchvision
import torchvision.transforms.functional as F
import seaborn as sns
import csv
from tqdm import tqdm

from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.nn import functional as F

# Huggingface
from huggingface_hub import hf_hub_url, hf_hub_download

# Real-ESRGAN
from real_esrgan.rrdbnet_arch import RRDBNet
from real_esrgan.utils import pad_reflect, split_image_into_overlapping_patches, stich_together, \
                   unpad_image

# VQA
from transformers import ViltProcessor, ViltForQuestionAnswering

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

## Anisotopric Diffusion
* Anisotropic diffusion is a image filtering known to remove noise in an image by smoothing the flat areas while preserving its edges.

In [ ]:
# Anisotropic Diffusion
def anisotropic(input_img, alpha=0.1, K=15, iterations=10, option=1):
        """Purifying image using anisotropic diffusion
        Args:
            input_img: Input image to be purified
            alpha: Conduction coefficient prevent over-smoothing
            K: Sensitivity to edges
            iterations: Number of iterations, chosen to provide sufficient denoising without computational overhead
            option: 1 for Perona-Malik (more aggressive), 2 for Tukey's biweight function (relies on region)
        Return:
            img: The purified image
        """
        img = input_img.astype(np.float32)

        for _ in range(iterations):
            # Computes differences between pixel and its neighbors (gradient)
            # axis = 0: vertical; axis = 1: horizontal
            north = np.roll(img, -1, axis=0) - img
            south = np.roll(img, 1, axis=0) - img
            east = np.roll(img, -1, axis=1) - img
            west = np.roll(img, 1, axis=1) - img

            # Conduction function
            # c_n, c_s, c_e, c_w: controls how much diffusion occurs in each direction
            # K: controls the sensitivity to edges
            # if small gradient == smoothen
            # if high gradient == edges, preserve
            if option == 1:
                c_n = np.exp(-(north/K)**2)
                c_s = np.exp(-(south/K)**2)
                c_e = np.exp(-(east/K)**2)
                c_w = np.exp(-(west/K)**2)
            else:
                c_n = 1.0 / (1.0 + (north/K)**2)
                c_s = 1.0 / (1.0 + (south/K)**2)
                c_e = 1.0 / (1.0 + (east/K)**2)
                c_w = 1.0 / (1.0 + (west/K)**2)

            # Update image
            # Blur the pixels based on the conduction values
            img += alpha * (c_n*north + c_s*south + c_e*east + c_w*west)

        # Return as image
        img = np.clip(img, 0, 255).astype(np.uint8)
        return img

## Real-ESRGAN
* This section applies Real-ESRGAN, a super-resolution for enhancing the image quality. This is used to finalize the purification process

In [ ]:
# Real-ESRGAN
HF_MODELS = {
    2: dict(
        repo_id='sberbank-ai/Real-ESRGAN',
        filename='RealESRGAN_x2.pth',
    ),
    4: dict(
        repo_id='sberbank-ai/Real-ESRGAN',
        filename='RealESRGAN_x4.pth',
    ),
    8: dict(
        repo_id='sberbank-ai/Real-ESRGAN',
        filename='RealESRGAN_x8.pth',
    ),
}

class RealESRGAN:
    def __init__(self, device, scale=2):
        self.device = device
        self.scale = scale
        self.model = RRDBNet(
            num_in_ch=3, num_out_ch=3, num_feat=64,
            num_block=23, num_grow_ch=32, scale=scale
        )

    def load_weights(self, model_path, download=True):
        if not os.path.exists(model_path) and download:
            assert self.scale in [2,4,8], 'You can download models only with scales: 2, 4, 8'
            config = HF_MODELS[self.scale]
            cache_dir = os.path.dirname(model_path)
            local_filename = os.path.basename(model_path)
            config_file_url = hf_hub_url(repo_id=config['repo_id'], filename=config['filename'])
            hf_hub_download(config_file_url, cache_dir=cache_dir, force_filename=local_filename)
            print('Weights downloaded to:', os.path.join(cache_dir, local_filename))

        loadnet = torch.load(model_path)
        if 'params' in loadnet:
            self.model.load_state_dict(loadnet['params'], strict=True)
        elif 'params_ema' in loadnet:
            self.model.load_state_dict(loadnet['params_ema'], strict=True)
        else:
            self.model.load_state_dict(loadnet, strict=True)
        self.model.eval()
        self.model.to(self.device)

    # @torch.amp.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu')
    def predict(self, lr_image, batch_size=4, patches_size=192,
                padding=24, pad_size=15):
        torch.set_num_threads(8)

        scale = self.scale
        device = self.device

        print("Enter predict")
        lr_image = np.array(lr_image)
        lr_image = pad_reflect(lr_image, pad_size)

        print("Split into patches")
        patches, p_shape = split_image_into_overlapping_patches(
            lr_image, patch_size=patches_size, padding_size=padding
        )
        print("Convert patches to tensor")
        img = torch.FloatTensor(patches/255).permute((0,3,1,2)).to(device)

        print("Process in batches")
        with torch.no_grad():
            print("Processing patches")
            res = self.model(img[0:batch_size])
            print("Reshape")
            for i in range(batch_size, img.shape[0], batch_size):
                print("Process patch")
                res = torch.cat((res, self.model(img[i:i+batch_size])), 0)

        print("Convert to image")
        sr_image = res.permute((0,2,3,1)).clamp_(0, 1).cpu().numpy()

        print("Stitch together")
        padded_size_scaled = tuple(np.multiply(p_shape[0:2], scale)) + (3,)
        scaled_image_shape = tuple(np.multiply(lr_image.shape[0:2], scale)) + (3,)
        sr_image = stich_together(
            sr_image, padded_image_shape=padded_size_scaled,
            target_shape=scaled_image_shape, padding_size=padding * scale
        )

        print("Unpad image")
        sr_img = unpad_image(sr_image, pad_size * scale)
        sr_img = (sr_img * 255).astype(np.uint8)
        sr_img = Image.fromarray(sr_img)

        print("Predict finished")
        return sr_img

In [ ]:
# Real-ESRGAN
def enhance(image: Image.Image):
    original_size = image.size

    model_path=CHECKPOINT_REALESRGAN

    upsampler = RealESRGAN(device=DEVICE, scale=scale)
    upsampler.load_weights(model_path, download=False)

    enhanced = upsampler.predict(image)
    restored = enhanced.resize(original_size, Image.LANCZOS)

    return restored

## Comparison on image
* This function will display the values on original image and clean image
* This will tell us that the image is purified

In [ ]:
def compare_image(original, purified):
    mse = np.mean((original.astype(np.float64) - purified.astype(np.float64)) ** 2)
    
    if mse == 0:
        return float("inf"), 1.0
            
    psnr_val = 20 * np.log10(255.0 / np.sqrt(mse))
    ssim_val = ssim(original, purified,data_range=255, channel_axis=2)

    return psnr_val, ssim_val

## Visual Question Answering
* This tool will helps us make final decision.
* Using the weighted average ensemble, we combine the detection from the detection and VQA. 

In [ ]:
# VQA
def get_answer(image, questions=QUESTIONS):
    """ Get answer based on the questions
    Args:
        image (PIL.Image): Input image in PIL format
        question (list): List of questions to ask the model
    Returns:
        answers (list): List of answers from the model
    """
    try:
        # img = _to_pil(image)
        print("Type vqa:", type(image))
        answers = []
        confidences = []

        for qa in questions:
            encoding = PROCESSOR(image, qa, return_tensors="pt") # Prepare inputs

            outputs = MODEL(**encoding)                   # Get model outputs
            logits = outputs.logits                       # Extract logits
            idx = logits.argmax(-1).item()                # Get index of highest logit
            answer = MODEL.config.id2label[idx]           # Get answers
            confidence = logits.softmax(-1).max().item()  # Get confidence score

            print(f"Q: {qa} A: {answer} (confidence: {confidence:.4f})")
            answers.append(answer)
            confidences.append(confidence)

        return answers, confidences
    except Exception as e:
        print("VQA error:", e)
        # Return empty lists if VQA fails
        return [], []

In [ ]:
# VQA
def vqa_conf(answers, vqa_confidences):
    """
    Calculate harmful score based on yes answers
    """
    harmful_confs = [conf for ans, conf in zip(answers[2:12], vqa_confidences[2:12]) if ans.lower() == "yes"]
    harmful_avg = np.mean(harmful_confs) if harmful_confs else 0.0
    print("Harmful score:", harmful_avg)

    # Identify art and educational context
    is_art = answers[0].lower() == "yes" and vqa_confidences[0] > 0.8
    is_educational = answers[1].lower() == "yes" and vqa_confidences[1] > 0.8

    if is_art:
        harmful_avg *= 0.5 # 50% reduction for art
    if is_educational:
        harmful_avg *= 0.5 # 50% reduction for educational

    return harmful_avg

In [ ]:
# VQA
def obj_detection_conf(detection_score, classes):
    """
    Calculate detection component with COCO class boost
    """
    # Boost score if there are detected classes
    coco_boost = 0.15 if any(name in COCO_CLASSES.values() for name in classes) else 0.0

    # Take maximum score from detection
    base_score = max(detection_score) if isinstance(detection_score, list) and detection_score else float(detection_score) if detection_score else 0.0

    # Make sure the final score does not exceed 1.0
    detection_component = min(base_score + coco_boost, 1.0)

    return detection_component

In [ ]:
# VQA
def decision(classes, answers, vqa_confidences, detection_score):
    """
    Make decision based on the answers and detected classes
    """
    # Calculate harmful score and boosts detection
    harmful_avg = vqa_conf(answers, vqa_confidences)
    detection_component = obj_detection_conf(detection_score, classes)

    # Weighted averaged based on VQA and Faster R-CNN confidence
    vqa_weight = 0.3 + (np.mean(vqa_confidences) * 0.2)  # between 0.3–0.5
    det_weight = 1.0 - vqa_weight
    print(f"Weights => VQA: {vqa_weight:.2f}, Detection: {det_weight:.2f}")

    # Weighted average ensemble
    total_score = (vqa_weight * harmful_avg) + (det_weight * detection_component)

    total_score = float(np.clip(total_score, 0.0, 1.0))

    print("Total score:", total_score)

    return total_score

## Detection
* This function process all images on test dataset.
* It also process the image using the previos functions.

In [ ]:
def detection(input_path, output_path, orig_path):
    if not os.path.exists(output_path):
        print("created")
        os.makedirs(output_path)

    filtered_count = 0
    image_count = 0
    data = []

    input_files = set(os.listdir(input_path))
    orig_files = set(os.listdir(orig_path))

    common_files = input_files.intersection(orig_files)
    
    for filename in common_files:
        if filename.lower().endswith(EXT):
            img_path = os.path.join(input_path, filename)
            orig_img_path = os.path.join(orig_path, filename)
            
            print(f"Processing: {filename}")
            
            input_img = Image.open(img_path).convert("RGB")
            orig_img = Image.open(orig_img_path).convert("RGB")
        
            # Purification
            purification = anisotropic(np.array(input_img))
            purification_pil = Image.fromarray(purification)
        
            # Real-ESRGAN
            enhance_img = enhance(purification_pil)

            # Comparison between images 
            # Case 1: Ground truth and purified
            # Case 2: Ground truth and attacked
            # Case 3: Attacked and purified
            psnr_trth_cln, ssim_trth_cln = compare_image(np.array(orig_img), np.array(enhance_img))
            psnr_trth_adv, ssim_trth_adv = compare_image(np.array(orig_img), np.array(input_img))
            psnr_adv_cln, ssim_adv_cln = compare_image(np.array(orig_img), np.array(input_img))
            # print(f"PSNR:{psnr_val}\nSSIM: {ssim_val}")
            
            # Output images
            print(f"output pat {output_path}")
            save_path = os.path.join(output_path, filename)
            enhance_img.save(save_path)
            print(f"Saved to {save_path}")

            # Predict
            outputs = predictor(np.array(enhance_img))

            # Classes
            classes = [COCO_CLASSES[int(cls)] for cls in outputs["instances"].pred_classes]
            scores = outputs["instances"].scores.tolist()
        
            # VQA
            answers, vqa_confidences = get_answer(enhance_img)
            total_score = decision(classes, answers, vqa_confidences, scores)

            image_count += 1

            is_filtered = "Yes" if total_score >= 0.8 else "No"
            
            if total_score >= 0.8:
                filtered_count += 1
                print(f"Image {os.path.basename(img_path)} is inappropriate {total_score:.2f}")

            data.append([filename, 
                         f"{psnr_trth_cln:.4f}",f"{ssim_trth_cln:.4f}", 
                         f"{psnr_trth_adv:.4f}", f"{ssim_trth_adv:.4f}", 
                         f"{psnr_adv_cln:.4f}", f"{ssim_adv_cln:.4f}",
                         f"{total_score:.2f}", is_filtered])
        
    print(f"Filtered count: {filtered_count}")
    return filtered_count, image_count, data

In [ ]:
orig_path = "/kaggle/working/lspd+hod-2/test/"

out_test_cln = "/kaggle/working/output_test_cln"
out_test_fgsm = "/kaggle/working/output_test_fgsm"

input_test_cln = "/kaggle/working/lspd+hod-2/test"
input_test_fgsm = "/kaggle/working/fgsm"

In [ ]:
register_coco_instances(
    "test_purif",
    {},
    "/kaggle/working/lspd+hod-2/test/_annotations.coco.json",
    "/kaggle/working/output_test_cln"
)

register_coco_instances(
    "test_purif_fgsm",
    {},
    "/kaggle/working/lspd+hod-2/test/_annotations.coco.json",
    "/kaggle/working/output_test_fgsm"
)

## Prediction, AP, and mAP

In [ ]:
filtered_count, image_count,data_cln = detection(input_test_cln, out_test_cln, orig_path)

clear_output(wait=True)
print(f"Filtered count: {filtered_count}\n Image Count:{image_count}")
print("Test clean successfully saved")

evaluator = COCOEvaluator("test_purif", cfg, False, output_dir="./output/")
test_loader = build_detection_test_loader(cfg, "test_purif")  # test dataset

results = inference_on_dataset(predictor.model, test_loader, evaluator)

map_purif_cln = results["bbox"]["AP"]

print(f"Validation Set Metrics\n{results}")
print(f"mAP: {map_purif_cln:.4f}")

ground_pred("test_purif")

In [ ]:
filtered_count, image_count,data_fgsm = detection(input_test_fgsm, out_test_fgsm, orig_path)

clear_output(wait=True)
print(f"Filtered count: {filtered_count}\n Image Count:{image_count}")
print("Test clean successfully saved")

evaluator = COCOEvaluator("test_purif_fgsm", cfg, False, output_dir="./output/")
test_loader = build_detection_test_loader(cfg, "test_purif_fgsm")  # test dataset
results = inference_on_dataset(predictor.model, test_loader, evaluator)
map_purif_fgsm = results["bbox"]["AP"]

print(f"Validation Set Metrics\n{results}")
print(f"mAP: {map_purif_fgsm:.4f}")

ground_pred("test_purif_fgsm")

## Confusion Matrix

In [ ]:
def calculate_iou(box1, box2):
    """Calculate IoU between two boxes in XYWH format"""
    # Convert to xyxy format
    box1_xyxy = [box1[0], box1[1], box1[0] + box1[2], box1[1] + box1[3]]
    box2_xyxy = [box2[0], box2[1], box2[0] + box2[2], box2[1] + box2[3]]

    # Calculate intersection
    x1 = max(box1_xyxy[0], box2_xyxy[0])
    y1 = max(box1_xyxy[1], box2_xyxy[1])
    x2 = min(box1_xyxy[2], box2_xyxy[2])
    y2 = min(box1_xyxy[3], box2_xyxy[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)

    # Calculate union
    box1_area = box1[2] * box1[3]
    box2_area = box2[2] * box2[3]
    union = box1_area + box2_area - intersection
    
    return intersection / union if union > 0 else 0

In [ ]:
def evaluate_detections(predictor, dataset_name, iou_threshold, score_threshold):
    """
    Evaluate object detection results and calculate confusion matrix metrics
    """
    dataset_dicts = DatasetCatalog.get(dataset_name)

    total_gt, tp, tn, fp, fn = 0, 0, 0, 0, 0

    all_conf = []
    all_tp = []
    
    print("Evaluating on dataset:", dataset_name)

    for d in tqdm(dataset_dicts):
        # Get ground truth boxes
        gt_boxes = [ann["bbox"] for ann in d["annotations"]]
        total_gt += len(gt_boxes)

        # Get predictions
        img = cv2.imread(d["file_name"])
        outputs = predictor(img)
        pred_boxes = outputs["instances"].pred_boxes.tensor.cpu().numpy()
        scores = outputs["instances"].scores.cpu().numpy()

        # Filter predictions by confidence threshold
        mask = scores >= score_threshold
        pred_boxes = pred_boxes[mask]
        scores = scores[mask]
        
        # Match predictions to ground truth
        matched_gt = set()

        for pred_idx, pred_box in enumerate(pred_boxes):
            best_iou = 0
            best_gt_idx = -1

            # Convert pred_box from xyxy to xywh format
            pred_box_xywh = [
                pred_box[0],
                pred_box[1],
                pred_box[2] - pred_box[0],
                pred_box[3] - pred_box[1]
            ]

            # Find best matching ground truth box
            for gt_idx, gt_box in enumerate(gt_boxes):
                if gt_idx in matched_gt:
                    continue
                    
                iou = calculate_iou(pred_box_xywh, gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx

            # Store confidence and whether it's a true positive
            all_conf.append(scores[pred_idx])
            
            if best_iou >= iou_threshold:
                tp += 1
                matched_gt.add(best_gt_idx)
                all_tp.append(1)
            else:
                fp += 1
                all_tp.append(0)

        # Count unmatched ground truth boxes as false negatives
        fn += len(gt_boxes) - len(matched_gt)
    
    # Calculate precision, recall, and F1 score
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
            
    # Create confusion matrix visualization
    plt.figure(figsize=(10, 8))
    confusion_matrix = np.array([[tn, fp], [fn, tp]])  # TN is always 0 in object detection
    sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'])
    plt.title(f'Confusion Matrix (IoU >= {iou_threshold}, Conf >= {score_threshold})')
    plt.ylabel('Ground Truth')
    plt.xlabel('Predicted')
    plt.close()

    # Print metrics
    print("\nDetection Metrics:")
    print(f"True Positives (TP): {tp}")
    print(f"True Negatives (TN): {tn}")
    print(f"False Positives (FP): {fp}")
    print(f"False Negatives (FN): {fn}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

    return tp, tn, fp, fn, accuracy, precision, recall, f1

In [ ]:
test_tp, test_tn, test_fp, test_fn, test_accuracy, test_precision, test_recall, test_f1 = evaluate_detections(
                    predictor, 
                    "test", 
                    iou_threshold=0.5, 
                    score_threshold=THRESHOLD)

test_fgsm_tp, test_fgsm_tn, test_fgsm_fp, test_fgsm_fn, test_fgsm_accuracy, test_fgsm_precision, test_fgsm_recall, test_fgsm_f1 = evaluate_detections(
                    predictor, 
                    "test_fgsm", 
                    iou_threshold=0.5, 
                    score_threshold=THRESHOLD)

In [ ]:
purif_tp, purif_tn, purif_fp, purif_fn, purif_accuracy, purif_precision, purif_recall, purif_f1 = evaluate_detections(
                    predictor, 
                    "test_purif", 
                    iou_threshold=0.5, 
                    score_threshold=THRESHOLD)

purif_fgsm_tp, purif_fgsm_tn, purif_fgsm_fp, purif_fgsm_fn, purif_fgsm_accuracy, purif_fgsm_precision, purif_fgsm_recall, purif_fgsm_f1 = evaluate_detections(
                    predictor, 
                    "test_purif_fgsm", 
                    iou_threshold=0.5, 
                    score_threshold=THRESHOLD)

In [ ]:
with open('data_cln.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['filename', 
                    'PSNR (GT vs Purif)', 'SSIM (GT vs Purif)', 
                    'PSNR (GT vs Adv)','SSIM (GT vs Adv)',
                    'PSNR (Adv vs Purif)', 'SSIM (Adv vs Purif)',
                    'total_score', 'filtered'])
    writer.writerows(data_cln)
    
with open('data_fgsm.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['filename', 
                    'PSNR (GT vs Purif)', 'SSIM (GT vs Purif)', 
                    'PSNR (GT vs Adv)','SSIM (GT vs Adv)',
                    'PSNR (Adv vs Purif)', 'SSIM (Adv vs Purif)',
                    'total_score', 'filtered'])
    writer.writerows(data_fgsm)

performance_data = [
    ['Metric', 'Without Purification', 'With Purification'],
    ['mAP Clean', f"{map_cln}", f"{map_purif_cln}"],
    ['mAP FGSM', f"{map_fgsm:.4f}", f"{map_purif_fgsm:.4f}"],
    ['Clean Accuracy', f"{test_accuracy: :.4f}", f"{purif_accuracy}"],
    ['Clean Precision', f"{test_precision: :.4f}", f"{purif_precision}"],
    ['Clean Recall', f"{test_recall: :.4f}", f"{purif_recall}"],
    ['Clean F1 Score', f"{test_f1: :.4f}", f"{purif_f1}"],
    ['Clean True Positive', f"{test_tp: :.4f}", f"{purif_tp}"],
    ['Clean True Negative', f"{test_tn: :.4f}", f"{purif_tn}"],
    ['Clean False Positive', f"{test_fp: :.4f}", f"{purif_fp}"],
    ['Clean False Negative', f"{test_fn: :.4f}", f"{purif_fn}"],
    ['FGSM Accuracy', f"{test_fgsm_accuracy:.4f}", f"{purif_fgsm_accuracy:.4f}"],
    ['FGSM Precision', f"{test_fgsm_precision:.4f}", f"{purif_fgsm_precision:.4f}"],
    ['FGSM Recall', f"{test_fgsm_recall:.4f}", f"{purif_fgsm_recall:.4f}"],
    ['FGSM F1', f"{test_fgsm_f1:.4f}", f"{purif_fgsm_f1:.4f}"],
    ['FGSM True Positve', f"{test_fgsm_tp}", f"{purif_fgsm_tp}"],
    ['FGSM True Negative', f"{test_fgsm_tn}", f"{purif_fgsm_tn}"],
    ['FGSM False Positve', f"{test_fgsm_fp}", f"{purif_fgsm_fp}"],
    ['FGSM False Negative', f"{test_fgsm_fn}", f"{purif_fgsm_fn}"]
]

with open('performance.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(performance_data)

print("Performance results saved to performance.csv")

# Visualization
* This section shows the comparison between the performance of Faster R-CNN with attack data and the purified attack data.

In [ ]:
labels = ["mAP", "Accuracy", "Precision", "Recall", "F1 Score"]

cln = [map_cln, 
       test_accuracy * 100, 
       test_precision * 100, 
       test_recall * 100, 
       test_f1 * 100]
cln_purif = [map_purif_cln, 
             purif_accuracy * 100, 
             purif_precision * 100, 
             purif_recall * 100, 
             purif_f1 * 100]
fgsm = [map_fgsm, 
        test_fgsm_accuracy * 100, 
        test_fgsm_precision * 100, 
        test_fgsm_recall * 100, 
        test_fgsm_f1 * 100]
fgsm_purif = [map_purif_fgsm, 
              purif_fgsm_accuracy * 100, 
              purif_fgsm_precision * 100, 
              purif_fgsm_recall * 100, 
              purif_fgsm_f1 * 100]

x = np.arange(len(labels))
width = 0.18

# Plot bars
plt.bar(x - 1.5*width, cln, width, label="Clean (Faster R-CNN)")
plt.bar(x - 0.5*width, cln_purif, width, label="Clean + Purification")
plt.bar(x + 0.5*width, fgsm, width, label="FGSM (Faster R-CNN)")
plt.bar(x + 1.5*width, fgsm_purif, width, label="FGSM + Purification")

# Formatting
plt.xticks(x, labels)
plt.xlabel("Performance Metrics")
plt.ylabel("Score")
plt.title("Performance Comparison With and Without Purification")
plt.ylim(0, 100)
plt.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
sys.exit()